# Phase 2: Stochastic Modelling & Derivatives Pricing  
## Notebook 02_08 — Dynamic Delta Hedging Simulation

### Research question

How does Black-Scholes-Merton delta hedging behave when hedging is discrete, realised volatility differs from implied volatility, transaction costs are present, or the underlying jumps?

This notebook connects the mathematical replication argument from Black-Scholes-Merton to the trading reality of dynamic hedging.

It covers:

1. Black-Scholes-Merton price, Delta, Gamma, Vega, and Theta;
2. risk-neutral GBM simulation;
3. discrete delta-hedging of a short option position;
4. self-financing cash account mechanics;
5. hedging error distribution;
6. effect of rebalancing frequency;
7. effect of realised volatility versus implied volatility;
8. transaction-cost drag;
9. jump-risk hedging failure;
10. Gamma/Theta intuition;
11. path-level PnL attribution;
12. saved hedging diagnostics and stress reports.

The core idea is:

> Continuous-time replication is an idealisation. Real hedging is discrete, costly, and exposed to model error.

## 1. The theoretical hedge

In the Black-Scholes-Merton framework, a European option value is:

$$
V(t,S_t)
$$
The option Delta is:

$$
\Delta_t = \frac{\partial V}{\partial S}(t,S_t)
$$
For a **short call**, the option seller can hedge by holding:

$$
+\Delta_t
$$
shares of the underlying.

For a **long call**, the holder can hedge by holding:

$$
-\Delta_t
$$
shares.

The theoretical BSM replication assumes:

1. continuous rebalancing;
2. no transaction costs;
3. no bid-ask spread;
4. no market impact;
5. constant volatility;
6. continuous sample paths;
7. unlimited borrowing and lending at the risk-free rate.

This notebook relaxes several of these assumptions computationally.

## 2. Why discrete hedging creates error

Under the BSM assumptions, the option can be replicated continuously.

But in practice, hedging occurs at discrete times:

$$
0=t_0<t_1<\dots<t_N=T
$$
The hedge uses $\Delta_{t_i}$ over the interval $[t_i,t_{i+1})$.

If the underlying moves substantially between rehedges, the hedge is no longer exact.

The error is strongly related to Gamma exposure:

$$
\Gamma_t = \frac{\partial^2 V}{\partial S^2}(t,S_t)
$$
Short options are typically short Gamma, meaning large underlying moves hurt the hedged position.

A rough intuition for short option hedging PnL over a small interval is:

$$
\begin{aligned}
\text{hedging PnL} &\approx \Theta \Delta t \\
&\quad + \frac{1}{2}\Gamma S^2 (\sigma_{\text{realised}}^2-\sigma_{\text{implied}}^2)\Delta t
\end{aligned}
$$
This is not the full accounting identity, but it captures the main volatility mismatch intuition:

- if you sell implied volatility above realised volatility, delta-hedged short-option PnL tends to be favourable;
- if realised volatility exceeds implied volatility, short-option hedging tends to lose.

## 3. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from math import erf, exp, log, pi, sqrt
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

In [ ]:
@dataclass(frozen=True)
class HedgingConfig:
    """
    Dynamic delta hedging simulation configuration.

    Parameters
    ----------
    s0:
        Initial underlying price.

    strike:
        Option strike.

    maturity:
        Option maturity in years.

    risk_free_rate:
        Continuously compounded risk-free rate.

    dividend_yield:
        Continuous dividend yield.

    implied_vol:
        Volatility used to price and hedge the option.

    realised_vol:
        Volatility used to simulate the underlying path.

    n_paths:
        Number of Monte Carlo paths.

    hedge_steps:
        Number of hedge rebalancing intervals.

    option_type:
        "call" or "put".

    position:
        "short_option" or "long_option".

    transaction_cost_bps:
        Proportional transaction cost in basis points per traded notional.

    seed:
        Random seed.
    """
    s0: float = 100.0
    strike: float = 100.0
    maturity: float = 1.0
    risk_free_rate: float = 0.04
    dividend_yield: float = 0.00
    implied_vol: float = 0.20
    realised_vol: float = 0.20
    n_paths: int = 80_000
    hedge_steps: int = 252
    option_type: str = "call"
    position: str = "short_option"
    transaction_cost_bps: float = 0.0
    seed: int = 42


base_config = HedgingConfig()
base_config

## 4. Black-Scholes-Merton price and Greeks

For a call:

$$
C = Se^{-q\tau}N(d_1)-Ke^{-r\tau}N(d_2)
$$
For a put:

$$
P = Ke^{-r\tau}N(-d_2)-Se^{-q\tau}N(-d_1)
$$
where:

$$
\begin{aligned}
d_1 &= \frac{ \log(S/K)+(r-q+\frac{1}{2}\sigma^2)\tau } {\sigma\sqrt{\tau}}
\end{aligned}
$$
$$
d_2=d_1-\sigma\sqrt{\tau}
$$
The Greeks used in the hedge are:

$$
\Delta_C=e^{-q\tau}N(d_1)
$$
$$
\Delta_P=e^{-q\tau}(N(d_1)-1)
$$
$$
\begin{aligned}
\Gamma &= \frac{e^{-q\tau}\phi(d_1)} {S\sigma\sqrt{\tau}}
\end{aligned}
$$
$$
\text{Vega}=Se^{-q\tau}\phi(d_1)\sqrt{\tau}
$$

In [ ]:
def normal_cdf(x: float | np.ndarray) -> float | np.ndarray:
    """
    Standard normal CDF.
    """
    x = np.asarray(x)
    return 0.5 * (1.0 + np.vectorize(erf)(x / np.sqrt(2.0)))


def normal_pdf(x: float | np.ndarray) -> float | np.ndarray:
    """
    Standard normal PDF.
    """
    x = np.asarray(x)
    return np.exp(-0.5 * x ** 2) / np.sqrt(2.0 * np.pi)


def bsm_d1_d2(
    spot: np.ndarray | float,
    strike: float,
    tau: np.ndarray | float,
    risk_free_rate: float,
    dividend_yield: float,
    volatility: float
) -> tuple[np.ndarray, np.ndarray]:
    """
    Compute BSM d1 and d2.

    Handles tau close to zero by clipping to a small positive value.
    """
    spot = np.asarray(spot, dtype=float)
    tau = np.asarray(tau, dtype=float)

    tau_safe = np.maximum(tau, 1e-12)

    d1 = (
        np.log(spot / strike)
        + (risk_free_rate - dividend_yield + 0.5 * volatility ** 2) * tau_safe
    ) / (volatility * np.sqrt(tau_safe))

    d2 = d1 - volatility * np.sqrt(tau_safe)

    return d1, d2


def bsm_price(
    spot: np.ndarray | float,
    strike: float,
    tau: np.ndarray | float,
    risk_free_rate: float,
    dividend_yield: float,
    volatility: float,
    option_type: str
) -> np.ndarray:
    """
    BSM price for call or put.
    """
    spot = np.asarray(spot, dtype=float)
    tau = np.asarray(tau, dtype=float)

    intrinsic_call = np.maximum(spot - strike, 0.0)
    intrinsic_put = np.maximum(strike - spot, 0.0)

    if np.all(tau <= 0):
        if option_type == "call":
            return intrinsic_call
        if option_type == "put":
            return intrinsic_put
        raise ValueError("option_type must be 'call' or 'put'.")

    d1, d2 = bsm_d1_d2(
        spot=spot,
        strike=strike,
        tau=tau,
        risk_free_rate=risk_free_rate,
        dividend_yield=dividend_yield,
        volatility=volatility
    )

    discounted_spot = spot * np.exp(-dividend_yield * tau)
    discounted_strike = strike * np.exp(-risk_free_rate * tau)

    if option_type == "call":
        value = discounted_spot * normal_cdf(d1) - discounted_strike * normal_cdf(d2)
        return np.where(tau <= 0, intrinsic_call, value)

    if option_type == "put":
        value = discounted_strike * normal_cdf(-d2) - discounted_spot * normal_cdf(-d1)
        return np.where(tau <= 0, intrinsic_put, value)

    raise ValueError("option_type must be 'call' or 'put'.")


def bsm_delta(
    spot: np.ndarray | float,
    strike: float,
    tau: np.ndarray | float,
    risk_free_rate: float,
    dividend_yield: float,
    volatility: float,
    option_type: str
) -> np.ndarray:
    """
    BSM Delta for call or put.
    """
    spot = np.asarray(spot, dtype=float)
    tau = np.asarray(tau, dtype=float)

    d1, _ = bsm_d1_d2(
        spot=spot,
        strike=strike,
        tau=tau,
        risk_free_rate=risk_free_rate,
        dividend_yield=dividend_yield,
        volatility=volatility
    )

    if option_type == "call":
        delta = np.exp(-dividend_yield * tau) * normal_cdf(d1)
        return np.where(tau <= 0, (spot > strike).astype(float), delta)

    if option_type == "put":
        delta = np.exp(-dividend_yield * tau) * (normal_cdf(d1) - 1.0)
        return np.where(tau <= 0, -(spot < strike).astype(float), delta)

    raise ValueError("option_type must be 'call' or 'put'.")


def bsm_gamma(
    spot: np.ndarray | float,
    strike: float,
    tau: np.ndarray | float,
    risk_free_rate: float,
    dividend_yield: float,
    volatility: float
) -> np.ndarray:
    """
    BSM Gamma.
    """
    spot = np.asarray(spot, dtype=float)
    tau = np.asarray(tau, dtype=float)
    tau_safe = np.maximum(tau, 1e-12)

    d1, _ = bsm_d1_d2(
        spot=spot,
        strike=strike,
        tau=tau_safe,
        risk_free_rate=risk_free_rate,
        dividend_yield=dividend_yield,
        volatility=volatility
    )

    gamma = (
        np.exp(-dividend_yield * tau_safe)
        * normal_pdf(d1)
        / (spot * volatility * np.sqrt(tau_safe))
    )

    return np.where(tau <= 0, 0.0, gamma)


def bsm_vega(
    spot: np.ndarray | float,
    strike: float,
    tau: np.ndarray | float,
    risk_free_rate: float,
    dividend_yield: float,
    volatility: float
) -> np.ndarray:
    """
    BSM Vega.
    """
    spot = np.asarray(spot, dtype=float)
    tau = np.asarray(tau, dtype=float)
    tau_safe = np.maximum(tau, 1e-12)

    d1, _ = bsm_d1_d2(
        spot=spot,
        strike=strike,
        tau=tau_safe,
        risk_free_rate=risk_free_rate,
        dividend_yield=dividend_yield,
        volatility=volatility
    )

    vega = spot * np.exp(-dividend_yield * tau_safe) * normal_pdf(d1) * np.sqrt(tau_safe)

    return np.where(tau <= 0, 0.0, vega)

In [ ]:
initial_price = float(bsm_price(
    spot=base_config.s0,
    strike=base_config.strike,
    tau=base_config.maturity,
    risk_free_rate=base_config.risk_free_rate,
    dividend_yield=base_config.dividend_yield,
    volatility=base_config.implied_vol,
    option_type=base_config.option_type
))

initial_delta = float(bsm_delta(
    spot=base_config.s0,
    strike=base_config.strike,
    tau=base_config.maturity,
    risk_free_rate=base_config.risk_free_rate,
    dividend_yield=base_config.dividend_yield,
    volatility=base_config.implied_vol,
    option_type=base_config.option_type
))

initial_gamma = float(bsm_gamma(
    spot=base_config.s0,
    strike=base_config.strike,
    tau=base_config.maturity,
    risk_free_rate=base_config.risk_free_rate,
    dividend_yield=base_config.dividend_yield,
    volatility=base_config.implied_vol
))

pd.Series({
    "initial_option_price": initial_price,
    "initial_delta": initial_delta,
    "initial_gamma": initial_gamma
})

## 5. Simulating underlying paths

We simulate the underlying under the **realised** volatility:

$$
dS_t = \mu S_tdt + \sigma_{\text{realised}}S_tdW_t
$$
For hedging experiments, the drift has much less influence than volatility and jumps over short horizons. We use risk-neutral drift for consistency:

$$
\mu = r-q
$$
The option is priced and hedged using **implied volatility**.

This creates a clean experiment:

- path dynamics use $\sigma_{\text{realised}}$;
- hedge model uses $\sigma_{\text{implied}}$.

In [ ]:
def simulate_gbm_paths(
    config: HedgingConfig,
    seed: int | None = None
) -> tuple[np.ndarray, np.ndarray]:
    """
    Simulate GBM paths using realised volatility.
    """
    if seed is None:
        seed = config.seed

    local_rng = np.random.default_rng(seed)

    n_paths = config.n_paths
    n_steps = config.hedge_steps
    dt = config.maturity / n_steps

    time_grid = np.linspace(0.0, config.maturity, n_steps + 1)

    z = local_rng.standard_normal(size=(n_paths, n_steps))

    log_increments = (
        (config.risk_free_rate - config.dividend_yield - 0.5 * config.realised_vol ** 2) * dt
        + config.realised_vol * sqrt(dt) * z
    )

    log_paths = np.zeros((n_paths, n_steps + 1))
    log_paths[:, 0] = log(config.s0)
    log_paths[:, 1:] = log(config.s0) + np.cumsum(log_increments, axis=1)

    return time_grid, np.exp(log_paths)

In [ ]:
time_grid, sample_paths = simulate_gbm_paths(
    config=HedgingConfig(n_paths=2_000, hedge_steps=base_config.hedge_steps),
    seed=123
)

plt.figure(figsize=(12, 6))
for i in range(50):
    plt.plot(time_grid, sample_paths[i], alpha=0.35)

plt.title("Simulated Underlying Paths")
plt.xlabel("Time")
plt.ylabel("Underlying price")
plt.show()

## 6. Delta-hedging accounting

We simulate a **short option** position.

At $t=0$:

1. sell the option and receive premium $V_0$;
2. buy $\Delta_0$ shares to hedge;
3. finance the remaining amount in the cash account.

Cash after setting up the hedge:

$$
\text{cash}_0 = V_0 - \Delta_0S_0 - \text{transaction cost}
$$
At each rebalancing time:

1. cash accrues interest;
2. compute new Delta;
3. trade $\Delta_{\text{new}}-\Delta_{\text{old}}$ shares;
4. subtract transaction costs;
5. hold hedge to the next date.

At maturity, the short option pays:

$$
-g(S_T)
$$
Final hedged PnL:

$$
\begin{aligned}
\Pi_T &= \Delta_T S_T \\
&\quad + \text{cash}_T \\
&\quad - g(S_T)
\end{aligned}
$$
For a long option, signs reverse.

In [ ]:
def option_payoff(spot: np.ndarray, strike: float, option_type: str) -> np.ndarray:
    """
    Vanilla option payoff.
    """
    if option_type == "call":
        return np.maximum(spot - strike, 0.0)

    if option_type == "put":
        return np.maximum(strike - spot, 0.0)

    raise ValueError("option_type must be 'call' or 'put'.")


def simulate_delta_hedge(
    config: HedgingConfig,
    return_path_diagnostics: bool = False
) -> tuple[pd.DataFrame, pd.DataFrame | None]:
    """
    Simulate dynamic delta hedging of a vanilla option.

    The default position is short option hedged with +Delta shares.
    """
    time_grid, paths = simulate_gbm_paths(config, seed=config.seed)

    n_paths, n_cols = paths.shape
    n_steps = n_cols - 1
    dt = config.maturity / n_steps
    tc_rate = config.transaction_cost_bps / 10_000.0

    option_sign = -1.0 if config.position == "short_option" else 1.0
    hedge_sign = -option_sign

    tau0 = config.maturity

    initial_option_value = bsm_price(
        spot=config.s0,
        strike=config.strike,
        tau=tau0,
        risk_free_rate=config.risk_free_rate,
        dividend_yield=config.dividend_yield,
        volatility=config.implied_vol,
        option_type=config.option_type
    )

    initial_delta_model = bsm_delta(
        spot=config.s0,
        strike=config.strike,
        tau=tau0,
        risk_free_rate=config.risk_free_rate,
        dividend_yield=config.dividend_yield,
        volatility=config.implied_vol,
        option_type=config.option_type
    )

    stock_position = hedge_sign * initial_delta_model * np.ones(n_paths)
    trade_notional = np.abs(stock_position) * config.s0
    transaction_cost = tc_rate * trade_notional

    # Short option: receive premium; Long option: pay premium.
    cash = -option_sign * initial_option_value - stock_position * config.s0 - transaction_cost

    total_transaction_cost = transaction_cost.copy()
    total_abs_rehedge_notional = trade_notional.copy()
    max_abs_delta = np.abs(stock_position).copy()

    diagnostic_rows = []

    if return_path_diagnostics:
        diagnostic_rows.append({
            "time_step": 0,
            "time": 0.0,
            "mean_spot": config.s0,
            "mean_model_delta": float(initial_delta_model),
            "mean_stock_position": float(np.mean(stock_position)),
            "mean_cash": float(np.mean(cash)),
            "mean_transaction_cost_cumulative": float(np.mean(total_transaction_cost))
        })

    for step in range(1, n_steps + 1):
        t = time_grid[step]
        tau = max(config.maturity - t, 0.0)

        # Cash accrues over the interval.
        cash *= exp(config.risk_free_rate * dt)

        spot_now = paths[:, step]

        if step < n_steps:
            new_delta_model = bsm_delta(
                spot=spot_now,
                strike=config.strike,
                tau=tau,
                risk_free_rate=config.risk_free_rate,
                dividend_yield=config.dividend_yield,
                volatility=config.implied_vol,
                option_type=config.option_type
            )

            new_stock_position = hedge_sign * new_delta_model

            shares_to_trade = new_stock_position - stock_position
            trade_notional = np.abs(shares_to_trade) * spot_now
            transaction_cost = tc_rate * trade_notional

            cash -= shares_to_trade * spot_now
            cash -= transaction_cost

            stock_position = new_stock_position

            total_transaction_cost += transaction_cost
            total_abs_rehedge_notional += trade_notional
            max_abs_delta = np.maximum(max_abs_delta, np.abs(stock_position))

        if return_path_diagnostics and step % max(1, n_steps // 20) == 0:
            diagnostic_rows.append({
                "time_step": step,
                "time": t,
                "mean_spot": float(np.mean(spot_now)),
                "mean_model_delta": float(np.mean(stock_position * hedge_sign)),
                "mean_stock_position": float(np.mean(stock_position)),
                "mean_cash": float(np.mean(cash)),
                "mean_transaction_cost_cumulative": float(np.mean(total_transaction_cost))
            })

    terminal_spot = paths[:, -1]
    payoff = option_payoff(terminal_spot, config.strike, config.option_type)

    final_portfolio_value = stock_position * terminal_spot + cash + option_sign * payoff

    # A positive PnL means the full hedged position made money.
    pnl = final_portfolio_value

    realised_log_returns = np.diff(np.log(paths), axis=1)
    realised_variance = np.sum(realised_log_returns ** 2, axis=1)
    realised_vol_path = np.sqrt(realised_variance / config.maturity)

    summary = pd.DataFrame({
        "pnl": pnl,
        "terminal_spot": terminal_spot,
        "payoff": payoff,
        "initial_option_value": initial_option_value,
        "total_transaction_cost": total_transaction_cost,
        "total_abs_rehedge_notional": total_abs_rehedge_notional,
        "max_abs_stock_position": max_abs_delta,
        "realised_vol_path": realised_vol_path,
        "implied_vol": config.implied_vol,
        "realised_vol_input": config.realised_vol,
        "hedge_steps": config.hedge_steps,
        "transaction_cost_bps": config.transaction_cost_bps,
        "position": config.position,
        "option_type": config.option_type
    })

    diagnostics = pd.DataFrame(diagnostic_rows) if return_path_diagnostics else None

    return summary, diagnostics

In [ ]:
hedge_summary, hedge_diagnostics = simulate_delta_hedge(
    base_config,
    return_path_diagnostics=True
)

hedge_summary.head()

## 7. Hedging PnL diagnostics

A good simulation should report the full distribution of hedging error, not only the mean.

We compute:

- mean PnL;
- standard deviation;
- skew;
- excess kurtosis;
- 1%, 5%, 50%, 95%, 99% quantiles;
- average transaction cost.

In [ ]:
def pnl_diagnostics(summary: pd.DataFrame, label: str) -> pd.Series:
    """
    Compute PnL distribution diagnostics.
    """
    pnl = summary["pnl"].to_numpy()
    mean = np.mean(pnl)
    std = np.std(pnl, ddof=1)
    centered = pnl - mean

    skew = np.mean(centered ** 3) / (std ** 3) if std > 0 else np.nan
    excess_kurtosis = np.mean(centered ** 4) / (std ** 4) - 3 if std > 0 else np.nan

    return pd.Series({
        "experiment": label,
        "n_paths": len(summary),
        "mean_pnl": mean,
        "std_pnl": std,
        "skew_pnl": skew,
        "excess_kurtosis_pnl": excess_kurtosis,
        "p01": np.quantile(pnl, 0.01),
        "p05": np.quantile(pnl, 0.05),
        "p50": np.quantile(pnl, 0.50),
        "p95": np.quantile(pnl, 0.95),
        "p99": np.quantile(pnl, 0.99),
        "mean_transaction_cost": summary["total_transaction_cost"].mean(),
        "mean_realised_vol_path": summary["realised_vol_path"].mean()
    })


base_pnl_diag = pnl_diagnostics(hedge_summary, "base_bsm_discrete_hedge")

base_pnl_diag

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(hedge_summary["pnl"], bins=100)
plt.axvline(0, linestyle="--")
plt.title("Delta-Hedged PnL Distribution")
plt.xlabel("PnL")
plt.ylabel("Count")
plt.show()

## 8. Rebalancing frequency experiment

As the number of hedge steps increases, hedging error should generally shrink under ideal BSM assumptions.

We compare:

- monthly;
- weekly;
- daily;
- twice daily;
- four times daily.

In a frictionless BSM world, more frequent hedging improves replication.

With transaction costs, more frequent hedging can reduce model error but increase cost.

In [ ]:
def rebalancing_frequency_experiment(
    base_config: HedgingConfig,
    step_grid: list[int],
    transaction_cost_bps: float = 0.0
) -> pd.DataFrame:
    """
    Run hedging simulations across different rebalancing frequencies.
    """
    rows = []

    for steps in step_grid:
        cfg = HedgingConfig(
            **{
                **asdict(base_config),
                "hedge_steps": steps,
                "transaction_cost_bps": transaction_cost_bps,
                "seed": base_config.seed + steps
            }
        )

        summary, _ = simulate_delta_hedge(cfg)
        diag = pnl_diagnostics(summary, f"steps_{steps}")
        diag["hedge_steps"] = steps
        diag["transaction_cost_bps"] = transaction_cost_bps

        rows.append(diag)

    return pd.DataFrame(rows)


step_grid = [12, 26, 52, 126, 252, 504, 1008]

frequency_results_no_cost = rebalancing_frequency_experiment(
    base_config,
    step_grid=step_grid,
    transaction_cost_bps=0.0
)

frequency_results_no_cost

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(frequency_results_no_cost["hedge_steps"], frequency_results_no_cost["std_pnl"], marker="o")
plt.xscale("log")
plt.title("Hedging Error Standard Deviation vs Rebalancing Frequency")
plt.xlabel("Hedge steps")
plt.ylabel("PnL standard deviation")
plt.show()

## 9. Transaction cost experiment

Real hedging requires trading.

If proportional transaction costs are included, the cash account loses:

$$
c \cdot |\Delta_{\text{new}}-\Delta_{\text{old}}|S_t
$$
where $c$ is the transaction cost rate.

More frequent hedging usually increases total turnover and transaction costs.

In [ ]:
frequency_results_cost = rebalancing_frequency_experiment(
    base_config,
    step_grid=step_grid,
    transaction_cost_bps=2.0
)

frequency_cost_comparison = pd.concat([
    frequency_results_no_cost.assign(cost_label="0 bps"),
    frequency_results_cost.assign(cost_label="2 bps")
], ignore_index=True)

frequency_cost_comparison[[
    "cost_label",
    "hedge_steps",
    "mean_pnl",
    "std_pnl",
    "mean_transaction_cost"
]]

In [ ]:
plt.figure(figsize=(10, 6))

for label, group in frequency_cost_comparison.groupby("cost_label"):
    plt.plot(group["hedge_steps"], group["mean_pnl"], marker="o", label=label)

plt.xscale("log")
plt.axhline(0, linestyle="--")
plt.title("Mean Hedged PnL vs Rebalancing Frequency with/without Costs")
plt.xlabel("Hedge steps")
plt.ylabel("Mean PnL")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(frequency_results_cost["hedge_steps"], frequency_results_cost["mean_transaction_cost"], marker="o")
plt.xscale("log")
plt.title("Mean Transaction Cost vs Rebalancing Frequency")
plt.xlabel("Hedge steps")
plt.ylabel("Mean cumulative transaction cost")
plt.show()

## 10. Realised versus implied volatility experiment

A delta-hedged short option position is strongly affected by the relationship between realised and implied volatility.

We hold implied volatility fixed and simulate paths with different realised volatilities.

For a short option:

- if realised volatility is below implied volatility, average hedged PnL tends to be positive;
- if realised volatility is above implied volatility, average hedged PnL tends to be negative.

This is the volatility risk premium intuition in a controlled toy setting.

In [ ]:
def realised_vs_implied_experiment(
    base_config: HedgingConfig,
    realised_vol_grid: list[float],
    transaction_cost_bps: float = 0.0
) -> pd.DataFrame:
    """
    Simulate hedging PnL as realised volatility changes while implied volatility is fixed.
    """
    rows = []

    for vol in realised_vol_grid:
        cfg = HedgingConfig(
            **{
                **asdict(base_config),
                "realised_vol": vol,
                "transaction_cost_bps": transaction_cost_bps,
                "seed": int(10_000 * vol) + 123
            }
        )

        summary, _ = simulate_delta_hedge(cfg)
        diag = pnl_diagnostics(summary, f"realised_vol_{vol:.2f}")
        diag["realised_vol"] = vol
        diag["implied_vol"] = cfg.implied_vol
        rows.append(diag)

    return pd.DataFrame(rows)


realised_vol_grid = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40]

vol_mismatch_results = realised_vs_implied_experiment(
    base_config,
    realised_vol_grid=realised_vol_grid,
    transaction_cost_bps=0.0
)

vol_mismatch_results[[
    "realised_vol",
    "implied_vol",
    "mean_pnl",
    "std_pnl",
    "p05",
    "p95"
]]

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(vol_mismatch_results["realised_vol"], vol_mismatch_results["mean_pnl"], marker="o")
plt.axhline(0, linestyle="--")
plt.axvline(base_config.implied_vol, linestyle="--", label="Implied vol used for hedge")
plt.title("Mean Delta-Hedged PnL vs Realised Volatility")
plt.xlabel("Realised volatility")
plt.ylabel("Mean PnL")
plt.legend()
plt.show()

## 11. PnL versus realised volatility by path

Even with the same input realised volatility, each path has its own realised variance.

For a short option, hedging PnL tends to decline when pathwise realised volatility is high.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(
    hedge_summary["realised_vol_path"],
    hedge_summary["pnl"],
    alpha=0.15,
    s=5
)
plt.axhline(0, linestyle="--")
plt.axvline(base_config.implied_vol, linestyle="--", label="Implied vol")
plt.title("Pathwise Hedged PnL vs Realised Volatility")
plt.xlabel("Pathwise realised volatility")
plt.ylabel("PnL")
plt.legend()
plt.show()

In [ ]:
pnl_vol_corr = hedge_summary["pnl"].corr(hedge_summary["realised_vol_path"])

pd.Series({
    "correlation_pnl_realised_vol": pnl_vol_corr,
    "position": base_config.position,
    "option_type": base_config.option_type
})

## 12. Gamma and near-expiry risk

Gamma is highest near the strike and near expiry.

This means delta can change very rapidly.

A short option hedger is most exposed when:

- the option is near-the-money;
- time to maturity is small;
- volatility is high;
- rebalancing is infrequent.

We visualise Gamma across spot prices and remaining maturities.

In [ ]:
spot_grid = np.linspace(60, 140, 161)
tau_grid = np.array([1.0, 0.5, 0.25, 0.10, 0.05, 0.02])

gamma_rows = []

for tau in tau_grid:
    gamma_values = bsm_gamma(
        spot=spot_grid,
        strike=base_config.strike,
        tau=tau,
        risk_free_rate=base_config.risk_free_rate,
        dividend_yield=base_config.dividend_yield,
        volatility=base_config.implied_vol
    )

    for spot, gamma in zip(spot_grid, gamma_values):
        gamma_rows.append({
            "spot": spot,
            "tau": tau,
            "gamma": gamma
        })

gamma_df = pd.DataFrame(gamma_rows)

gamma_df.head()

In [ ]:
plt.figure(figsize=(10, 6))

for tau, group in gamma_df.groupby("tau"):
    plt.plot(group["spot"], group["gamma"], label=f"tau={tau:.2f}")

plt.axvline(base_config.strike, linestyle="--", label="Strike")
plt.title("BSM Gamma Across Spot and Time to Maturity")
plt.xlabel("Spot")
plt.ylabel("Gamma")
plt.legend()
plt.show()

## 13. Jump-risk experiment

Delta hedging assumes continuous underlying paths.

A jump can move the underlying before the hedge has time to adjust.

We simulate paths with occasional jumps and hedge them using the BSM delta model that assumes no jumps.

This shows why jump risk is difficult to hedge with only the underlying.

In [ ]:
@dataclass(frozen=True)
class JumpHedgingConfig(HedgingConfig):
    """
    Hedging config with jump simulation parameters.
    """
    jump_intensity: float = 1.0
    jump_mean: float = -0.08
    jump_vol: float = 0.22


def simulate_jump_diffusion_paths_for_hedging(
    config: JumpHedgingConfig,
    seed: int | None = None
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Simulate jump-diffusion paths for hedging stress testing.

    Hedge model will still use BSM Delta with implied_vol.
    """
    if seed is None:
        seed = config.seed

    local_rng = np.random.default_rng(seed)

    n_paths = config.n_paths
    n_steps = config.hedge_steps
    dt = config.maturity / n_steps
    time_grid = np.linspace(0.0, config.maturity, n_steps + 1)

    kappa = exp(config.jump_mean + 0.5 * config.jump_vol ** 2) - 1.0

    z = local_rng.standard_normal(size=(n_paths, n_steps))
    jump_counts = local_rng.poisson(config.jump_intensity * dt, size=(n_paths, n_steps))

    jump_sums = np.zeros((n_paths, n_steps))

    jump_locations = np.where(jump_counts > 0)

    for path_idx, step_idx in zip(*jump_locations):
        count = jump_counts[path_idx, step_idx]
        jump_sums[path_idx, step_idx] = local_rng.normal(
            loc=config.jump_mean,
            scale=config.jump_vol,
            size=count
        ).sum()

    log_increments = (
        (
            config.risk_free_rate
            - config.dividend_yield
            - config.jump_intensity * kappa
            - 0.5 * config.realised_vol ** 2
        ) * dt
        + config.realised_vol * sqrt(dt) * z
        + jump_sums
    )

    log_paths = np.zeros((n_paths, n_steps + 1))
    log_paths[:, 0] = log(config.s0)
    log_paths[:, 1:] = log(config.s0) + np.cumsum(log_increments, axis=1)

    return time_grid, np.exp(log_paths), jump_counts

In [ ]:
def simulate_delta_hedge_given_paths(
    config: HedgingConfig,
    time_grid: np.ndarray,
    paths: np.ndarray
) -> pd.DataFrame:
    """
    Delta hedge using externally supplied paths.
    """
    n_paths, n_cols = paths.shape
    n_steps = n_cols - 1
    dt = config.maturity / n_steps
    tc_rate = config.transaction_cost_bps / 10_000.0

    option_sign = -1.0 if config.position == "short_option" else 1.0
    hedge_sign = -option_sign

    initial_option_value = bsm_price(
        spot=config.s0,
        strike=config.strike,
        tau=config.maturity,
        risk_free_rate=config.risk_free_rate,
        dividend_yield=config.dividend_yield,
        volatility=config.implied_vol,
        option_type=config.option_type
    )

    initial_delta_model = bsm_delta(
        spot=config.s0,
        strike=config.strike,
        tau=config.maturity,
        risk_free_rate=config.risk_free_rate,
        dividend_yield=config.dividend_yield,
        volatility=config.implied_vol,
        option_type=config.option_type
    )

    stock_position = hedge_sign * initial_delta_model * np.ones(n_paths)
    transaction_cost = tc_rate * np.abs(stock_position) * config.s0

    cash = -option_sign * initial_option_value - stock_position * config.s0 - transaction_cost
    total_transaction_cost = transaction_cost.copy()

    for step in range(1, n_steps + 1):
        cash *= exp(config.risk_free_rate * dt)

        spot_now = paths[:, step]
        tau = max(config.maturity - time_grid[step], 0.0)

        if step < n_steps:
            delta_model = bsm_delta(
                spot=spot_now,
                strike=config.strike,
                tau=tau,
                risk_free_rate=config.risk_free_rate,
                dividend_yield=config.dividend_yield,
                volatility=config.implied_vol,
                option_type=config.option_type
            )

            new_stock_position = hedge_sign * delta_model
            shares_to_trade = new_stock_position - stock_position
            transaction_cost = tc_rate * np.abs(shares_to_trade) * spot_now

            cash -= shares_to_trade * spot_now
            cash -= transaction_cost

            total_transaction_cost += transaction_cost
            stock_position = new_stock_position

    terminal_spot = paths[:, -1]
    payoff = option_payoff(terminal_spot, config.strike, config.option_type)
    pnl = stock_position * terminal_spot + cash + option_sign * payoff

    realised_log_returns = np.diff(np.log(paths), axis=1)
    realised_vol_path = np.sqrt(np.sum(realised_log_returns ** 2, axis=1) / config.maturity)

    return pd.DataFrame({
        "pnl": pnl,
        "terminal_spot": terminal_spot,
        "payoff": payoff,
        "total_transaction_cost": total_transaction_cost,
        "realised_vol_path": realised_vol_path
    })

In [ ]:
jump_config = JumpHedgingConfig(
    **{
        **asdict(base_config),
        "n_paths": 60_000,
        "realised_vol": 0.18,
        "implied_vol": 0.20,
        "jump_intensity": 1.0,
        "jump_mean": -0.08,
        "jump_vol": 0.22,
        "seed": 777
    }
)

jump_time_grid, jump_paths, jump_counts = simulate_jump_diffusion_paths_for_hedging(
    jump_config,
    seed=777
)

jump_hedge_summary = simulate_delta_hedge_given_paths(
    config=jump_config,
    time_grid=jump_time_grid,
    paths=jump_paths
)

gbm_comparable_config = HedgingConfig(
    **{
        **asdict(base_config),
        "n_paths": 60_000,
        "realised_vol": 0.18,
        "implied_vol": 0.20,
        "seed": 777
    }
)

gbm_comparable_summary, _ = simulate_delta_hedge(gbm_comparable_config)

jump_vs_gbm = pd.DataFrame([
    pnl_diagnostics(gbm_comparable_summary, "gbm_realised_vol_18"),
    pnl_diagnostics(jump_hedge_summary.assign(
        total_transaction_cost=jump_hedge_summary["total_transaction_cost"],
        hedge_steps=jump_config.hedge_steps,
        transaction_cost_bps=jump_config.transaction_cost_bps,
        position=jump_config.position,
        option_type=jump_config.option_type,
        implied_vol=jump_config.implied_vol,
        realised_vol_input=jump_config.realised_vol
    ), "jump_diffusion_stress")
])

jump_vs_gbm

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(gbm_comparable_summary["pnl"], bins=100, alpha=0.6, label="GBM")
plt.hist(jump_hedge_summary["pnl"], bins=100, alpha=0.5, label="Jump diffusion")
plt.axvline(0, linestyle="--")
plt.title("Delta-Hedged PnL: GBM vs Jump-Diffusion Stress")
plt.xlabel("PnL")
plt.ylabel("Count")
plt.legend()
plt.show()

## 14. Jump count and hedging loss relationship

We check whether paths with jumps produce worse hedging outcomes.

The exact relationship depends on jump direction, option type, and position.

For a short call, large upward jumps are especially dangerous.

In [ ]:
total_jumps_by_path = jump_counts.sum(axis=1)

jump_analysis = jump_hedge_summary.copy()
jump_analysis["n_jumps"] = total_jumps_by_path

jump_count_summary = (
    jump_analysis
    .groupby("n_jumps")
    .agg(
        n_paths=("pnl", "count"),
        mean_pnl=("pnl", "mean"),
        std_pnl=("pnl", "std"),
        p05=("pnl", lambda x: np.quantile(x, 0.05)),
        p50=("pnl", "median"),
        p95=("pnl", lambda x: np.quantile(x, 0.95))
    )
    .reset_index()
)

jump_count_summary.head(10)

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(jump_count_summary["n_jumps"], jump_count_summary["mean_pnl"])
plt.axhline(0, linestyle="--")
plt.title("Mean Hedged PnL by Number of Jumps")
plt.xlabel("Number of jumps over path")
plt.ylabel("Mean PnL")
plt.show()

## 15. Hedging report table

We collect the main experiments into one report:

1. base BSM discrete hedge;
2. rebalancing-frequency experiment;
3. transaction-cost experiment;
4. realised-volatility mismatch experiment;
5. jump-diffusion stress.

In [ ]:
report_rows = []

report_rows.append(base_pnl_diag)

for _, row in frequency_results_no_cost.iterrows():
    report_rows.append(row)

for _, row in frequency_results_cost.iterrows():
    report_rows.append(row)

for _, row in vol_mismatch_results.iterrows():
    report_rows.append(row)

report_rows.append(jump_vs_gbm.iloc[1])

hedging_report = pd.DataFrame(report_rows)

hedging_report.head()

## 16. Saving outputs

The notebook saves:

1. base hedging PnL sample;
2. base path diagnostics;
3. rebalancing-frequency results;
4. transaction-cost comparison;
5. realised-versus-implied volatility results;
6. jump-stress diagnostics;
7. jump-count PnL table;
8. gamma surface;
9. manifest.

These outputs can be used by later risk, execution, and option-surface notebooks.

In [ ]:
output_dir = Path("data/processed/dynamic_delta_hedging_simulation_v1")
output_dir.mkdir(parents=True, exist_ok=True)

base_summary_path = output_dir / "base_hedge_pnl_sample.csv"
base_diagnostics_path = output_dir / "base_hedge_time_diagnostics.csv"
frequency_no_cost_path = output_dir / "rebalancing_frequency_no_cost.csv"
frequency_cost_path = output_dir / "rebalancing_frequency_with_cost.csv"
vol_mismatch_path = output_dir / "realised_vs_implied_vol_experiment.csv"
jump_vs_gbm_path = output_dir / "jump_vs_gbm_hedging_summary.csv"
jump_count_path = output_dir / "jump_count_pnl_summary.csv"
gamma_surface_path = output_dir / "gamma_surface.csv"
hedging_report_path = output_dir / "hedging_report.csv"
manifest_path = output_dir / "manifest.json"

# Save sample of path-level PnL to keep output lightweight.
hedge_summary.sample(
    n=min(10_000, len(hedge_summary)),
    random_state=42
).to_csv(base_summary_path, index=False)

if hedge_diagnostics is not None:
    hedge_diagnostics.to_csv(base_diagnostics_path, index=False)

frequency_results_no_cost.to_csv(frequency_no_cost_path, index=False)
frequency_results_cost.to_csv(frequency_cost_path, index=False)
vol_mismatch_results.to_csv(vol_mismatch_path, index=False)
jump_vs_gbm.to_csv(jump_vs_gbm_path, index=False)
jump_count_summary.to_csv(jump_count_path, index=False)
gamma_df.to_csv(gamma_surface_path, index=False)
hedging_report.to_csv(hedging_report_path, index=False)

manifest = {
    "dataset_name": "dynamic_delta_hedging_simulation",
    "schema_version": "dynamic_delta_hedging_simulation_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "base_config": asdict(base_config),
    "jump_config": asdict(jump_config),
    "initial_option_price": initial_price,
    "initial_delta": initial_delta,
    "initial_gamma": initial_gamma,
    "base_mean_pnl": float(base_pnl_diag["mean_pnl"]),
    "base_std_pnl": float(base_pnl_diag["std_pnl"]),
    "notes": [
        "Base experiment hedges a short vanilla option with BSM delta.",
        "Underlying paths are simulated with realised volatility.",
        "Option is priced and hedged with implied volatility.",
        "Transaction costs are proportional to traded notional.",
        "Jump experiment hedges jump-diffusion paths using a BSM delta model."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

base_summary_path, frequency_no_cost_path, vol_mismatch_path, jump_vs_gbm_path, manifest_path

## 17. Practical checklist for delta-hedging simulations

Before trusting a hedging simulation, check:

1. **Position sign**  
   Are you simulating a short option or long option?

2. **Hedge sign**  
   Does the stock hedge offset the option Delta?

3. **Cash account**  
   Does cash accrue interest correctly?

4. **Self-financing mechanics**  
   Are rehedging trades funded from cash?

5. **Transaction costs**  
   Are costs proportional to traded notional, spread, or both?

6. **Implied versus realised volatility**  
   Which volatility prices the option, and which volatility drives paths?

7. **Rebalancing frequency**  
   How often is Delta updated?

8. **Terminal payoff**  
   Is the option payoff subtracted or added with the correct sign?

9. **Model mismatch**  
   Does the hedge model match the path model?

10. **Jumps**  
   Are discontinuous moves present, and can Delta hedging handle them?

11. **PnL distribution**  
   Are tail losses reported, not just mean PnL?

12. **Audit trail**  
   Are parameters and diagnostics saved?

## 18. Limitations

### 18.1 BSM assumptions are idealised

The base hedge assumes:

- constant volatility;
- continuous paths;
- no stochastic rates;
- no liquidity constraints;
- no market impact;
- no discrete tick sizes;
- no bid-ask dynamics.

### 18.2 Discrete hedging is model-dependent

The hedge uses BSM Delta.

If the true process has stochastic volatility, jumps, or fat tails, BSM Delta may be systematically wrong.

### 18.3 Transaction costs are simplified

The notebook uses proportional costs.

Real hedging costs include:

- bid-ask spread;
- market impact;
- latency;
- partial fills;
- borrow fees;
- funding costs;
- exchange fees;
- discrete contract multipliers.

### 18.4 Drift is not the main focus

The simulation uses risk-neutral drift.

For hedging-error experiments, volatility and path behaviour usually dominate drift, but real PnL attribution may require physical drift assumptions.

### 18.5 No volatility surface dynamics

The implied volatility used for hedging is constant.

Real option books are exposed to changes in implied volatility, skew, term structure, and volatility-of-volatility.

### 18.6 No Vega hedge

The hedge uses only the underlying.

Real desks often hedge Vega, skew, and higher-order Greeks with other options.

### 18.7 Jump risk cannot be perfectly delta hedged

A jump occurs before the hedge can adjust.

This is a structural limitation, not a numerical bug.

## 19. Research frontier and current directions

Dynamic hedging is still central in modern derivatives research because real markets violate the frictionless BSM assumptions.

### 19.1 Discrete hedging error asymptotics

Classical research studies how hedging error behaves as the rebalancing interval shrinks.

A future notebook could derive and empirically test the scaling of hedging-error variance with hedge frequency.

### 19.2 Transaction-cost-aware hedging

With transaction costs, continuous rebalancing is no longer optimal.

No-transaction bands and impulse-control methods can reduce unnecessary trading.

A future notebook could implement a Delta no-trade band and compare hedging error versus transaction cost.

### 19.3 Deep hedging

Deep hedging learns hedging strategies directly under frictions, risk measures, and constraints.

Instead of using analytical Delta, a neural network can choose hedge positions based on state variables.

A future notebook could compare BSM Delta hedging with a simple learned hedge under transaction costs.

### 19.4 Hedging under stochastic volatility

If volatility is stochastic, a Delta-only hedge does not fully hedge the option.

Vega risk remains.

A future notebook could simulate Heston paths and compare BSM Delta, Heston Delta, and Delta-Vega hedges.

### 19.5 Jump-risk hedging

Jump risk cannot be eliminated by continuous trading in the underlying.

A future notebook could study hedging with additional options or crash-protection instruments.

### 19.6 Leland volatility adjustment

Leland proposed modifying volatility to account for transaction costs in discrete hedging.

A future notebook could implement Leland's adjusted volatility and test whether it improves short-option hedging performance.

### 19.7 Market microstructure-aware hedging

Real rebalancing occurs through bid/ask quotes and order execution.

With L1 data, a later notebook could simulate hedging at `bid1`/`ask1` rather than mid-price.

## 20. Suggested follow-up notebooks

This notebook naturally leads to:

1. **02_11_heston_model_calibration**  
   Studying hedging and pricing when volatility is stochastic.

2. **02_Bates_stochastic_volatility_with_jumps**  
   Combining stochastic volatility and jumps.

3. **03_09_volatility_surface_alpha_signals**  
   Understanding how implied volatility changes affect hedged PnL.

4. **04_07_beta_weighted_tail_risk_hedging**  
   Using options to manage portfolio crash exposure.

5. **05_03_transaction_costs_slippage_latency**  
   Adding realistic execution costs to hedge rebalancing.

6. **06_deep_hedging_with_transaction_costs**  
   Learning hedging strategies under frictions.

7. **01_09_tick_to_bar_sampling_bias**  
   Connecting hedge execution to L1 bid/ask data.

8. **07_02_volatility_surface_pricing_and_alpha**  
   Combining option pricing, surface features, and hedging diagnostics.

## 21. Summary

This notebook simulated dynamic Delta hedging.

It showed how to:

1. compute BSM price, Delta, Gamma, and Vega;
2. simulate risk-neutral GBM paths;
3. hedge a short option using BSM Delta;
4. account for stock trades and cash account mechanics;
5. measure hedging error distribution;
6. study rebalancing frequency;
7. include transaction costs;
8. analyse realised-versus-implied volatility mismatch;
9. connect PnL to pathwise realised volatility;
10. visualise Gamma risk near expiry;
11. stress the hedge with jump-diffusion paths.

The key computational takeaway is:

> A dynamic hedge simulation is an accounting engine. Sign conventions, cash accrual, rebalancing trades, and transaction costs must be handled explicitly.

The key financial takeaway is:

> BSM replication is exact only under ideal assumptions. Discrete hedging, transaction costs, volatility mismatch, and jumps turn replication into risk management.

## 22. Further reading

### Classical foundations

- Black, F. and Scholes, M. "The Pricing of Options and Corporate Liabilities."
- Merton, R. C. "Theory of Rational Option Pricing."
- Hull, J. C. *Options, Futures, and Other Derivatives.*
- Shreve, S. E. *Stochastic Calculus for Finance II.*

### Hedging under frictions

- Leland, H. "Option Pricing and Replication with Transactions Costs."
- Whalley, A. E. and Wilmott, P. on optimal hedging with transaction costs.
- Literature on discrete hedging error.
- Literature on no-transaction bands and impulse control.

### Modern extensions

- Deep hedging.
- Hedging under stochastic volatility.
- Hedging jump risk.
- Delta-Vega hedging.
- Market microstructure-aware hedging.
- Reinforcement learning for hedging under transaction costs.